In [ ]:
#!pip install xgboost

In [ ]:
#!pip install cython 
#dependency for rankeval

In [ ]:
#!pip install rankeval

In [ ]:
#!pip install optuna

In [11]:
import pandas as pd
import numpy as np
import os

In [12]:
from sklearn.model_selection import GroupShuffleSplit

In [13]:
from sklearn.metrics import ndcg_score, f1_score
from scipy.stats import spearmanr

In [14]:
import xgboost as xgb

In [15]:
import optuna

In [16]:
from sklearn.model_selection import GroupKFold

In [17]:
import joblib

In [ ]:
data = pd.read_csv("data/data.csv")

In [19]:
data.head()

,Driver,session_id,season,Avg_Lap_Time,Avg_Sector_1_Time,Avg_Sector_2_Time,Avg_Sector_3_Time,Avg_Speed,Max_Speed,Avg_Throttle,...,Team_Ferrari,Team_Haas F1 Team,Team_McLaren,Team_Mercedes,Team_Red Bull Racing,Team_Williams,Team_Racing Bulls,Team_Kick Sauber,Team_Alpine,Team_Aston Martin
0,44,2019_1,2019,99.887000,34.987722,26.688277,38.263285,72.362914,321.0,77.332428,...,False,False,False,True,False,False,False,False,False,False
1,77,2019_1,2019,102.112642,34.605052,26.578421,41.077666,77.224164,322.0,63.533740,...,False,False,False,True,False,False,False,False,False,False
2,5,2019_1,2019,102.006363,35.413562,26.630250,39.076916,64.468105,318.0,65.327970,...,True,False,False,False,False,False,False,False,False,False
3,33,2019_1,2019,99.378250,35.317647,27.301647,36.972250,68.294922,320.0,45.247477,...,False,False,False,False,True,False,False,False,False,False
4,16,2019_1,2019,98.392416,34.423823,26.681176,38.623153,68.324947,321.0,72.722550,...,True,False,False,False,False,False,False,False,False,False


In [ ]:
#splliting data for training and testing
#split is grouped by session_id
gss = GroupShuffleSplit(test_size=.30, n_splits=1, random_state = 11).split(data, groups=data['session_id'])

X_train_splits, X_test_splits = next(gss)

#Final_Race_Position is the target variable
train_data = data.iloc[X_train_splits]
X_train = train_data.loc[:, ~train_data.columns.isin(['session_id', 'Final_Race_Position'])]
y_train = train_data.loc[:, train_data.columns.isin(['Final_Race_Position'])]

test_data = data.iloc[X_test_splits]
X_test = test_data.loc[:, ~test_data.columns.isin(['session_id', 'Final_Race_Position'])]
y_test = test_data.loc[:, test_data.columns.isin(['Final_Race_Position'])]

In [22]:
X_train.shape

(1764, 30)

In [23]:
X_test.shape

(774, 30)

In [ ]:
#each qualifying session is a separate group
groups = train_data.groupby('session_id').size().to_numpy()
groups

array([20, 20, 19, 20, 19, 18, 20, 20, 20, 20, 18, 20, 20, 20, 20, 20, 20,
       20, 20, 20, 20, 20, 20, 20, 20, 19, 20, 20, 20, 20, 20, 20, 20, 20,
       20, 19, 18, 19, 20, 20, 20, 20, 20, 20, 20, 20, 20, 18, 20, 20, 20,
       20, 20, 20, 20, 20, 20, 20, 20, 19, 19, 20, 20, 20, 20, 20, 20, 20,
       20, 20, 20, 20, 20, 20, 20, 20, 19, 20, 20, 20, 20, 20, 20, 20, 20,
       20, 20, 20, 20])

In [25]:
model = xgb.XGBRanker(
    tree_method = 'hist', #fastest algorithm
    booster = 'gbtree', #gradient boosting
    objective = 'rank:pairwise', #comparing pairs of drivers
    random_state = 42,
    learning_rate = 0.1,
    colsample_bytree = 0.9, #percentage of features used
    max_depth = 6, #depth of trees
    n_estimators = 100, 
    subsample = 0.75,
    eval_metric = 'ndcg'
)

model.fit(X_train, y_train, group=groups, verbose=True)

XGBRanker(base_score=None, booster='gbtree', callbacks=None,
          colsample_bylevel=None, colsample_bynode=None, colsample_bytree=0.9,
          device=None, early_stopping_rounds=None, enable_categorical=False,
          eval_metric='ndcg', feature_types=None, feature_weights=None,
          gamma=None, grow_policy=None, importance_type=None,
          interaction_constraints=None, learning_rate=0.1, max_bin=None,
          max_cat_threshold=None, max_cat_to_onehot=None, max_delta_step=None,
          max_depth=6, max_leaves=None, min_child_weight=None, missing=nan,
          monotone_constraints=None, multi_strategy=None, n_estimators=100,
          n_jobs=None, num_parallel_tree=None, ...)

In [26]:
#predicting by each race
#the predict function does not take a group argument

def predict(model, df):
    scores = model.predict(df.drop(columns=['session_id', 'Final_Race_Position']))
    predictions_df = df.copy()
    predictions_df['predicted_score'] = scores
    predictions_df['predicted_position'] = predictions_df['predicted_score'].rank(ascending=True, method='first')
    return predictions_df

predictions = (
    test_data.groupby('session_id')
    .apply(lambda x: predict(model, x))
    .reset_index(drop=True)
)

In [27]:
predictions

,Driver,session_id,season,Avg_Lap_Time,Avg_Sector_1_Time,Avg_Sector_2_Time,Avg_Sector_3_Time,Avg_Speed,Max_Speed,Avg_Throttle,...,Team_McLaren,Team_Mercedes,Team_Red Bull Racing,Team_Williams,Team_Racing Bulls,Team_Kick Sauber,Team_Alpine,Team_Aston Martin,predicted_score,predicted_position
0,44,2019_1,2019,99.887000,34.987722,26.688277,38.263285,72.362914,321.0,77.332428,...,False,True,False,False,False,False,False,False,-1.479586,1.0
1,77,2019_1,2019,102.112642,34.605052,26.578421,41.077666,77.224164,322.0,63.533740,...,False,True,False,False,False,False,False,False,-1.153539,2.0
2,5,2019_1,2019,102.006363,35.413562,26.630250,39.076916,64.468105,318.0,65.327970,...,False,False,False,False,False,False,False,False,-0.414978,5.0
3,33,2019_1,2019,99.378250,35.317647,27.301647,36.972250,68.294922,320.0,45.247477,...,False,False,True,False,False,False,False,False,-0.688962,3.0
4,16,2019_1,2019,98.392416,34.423823,26.681176,38.623153,68.324947,321.0,72.722550,...,False,False,False,False,False,False,False,False,-0.520811,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
769,2,2024_8,2024,86.852900,22.541250,39.724500,25.311800,31.497999,288.0,69.097964,...,False,False,False,True,False,False,False,False,0.571596,20.0
770,20,2024_8,2024,88.190000,25.223833,40.967666,28.480800,46.430601,294.0,67.484079,...,False,False,False,False,False,False,False,False,0.262595,18.0
771,11,2024_8,2024,88.602100,25.606333,39.694833,30.796272,31.072328,289.0,66.743170,...,False,False,True,False,False,False,False,False,0.061063,15.0
772,77,2024_8,2024,91.619666,24.757636,39.901818,26.462000,28.503915,289.0,70.254684,...,False,False,False,False,False,True,False,False,0.210623,16.0


In [28]:
#precision@k

def top_k_precision(predictions, k):
    precision = []

    for _, group in predictions.groupby('session_id'):
        actual_top_k = set(group.nsmallest(k, 'Final_Race_Position')['Driver']) #actual ranks
        predicted_top_k = set(group.nsmallest(k, 'predicted_position')['Driver']) #predicted ranks

        correct_predictions = len(actual_top_k & predicted_top_k) #predicted drivers that were actually in the top k
        precision.append(correct_predictions / k)

    return np.mean(precision) #average precision across races

print("Winner Precision:", top_k_precision(predictions, k=1))
print("Top 3 Precision:", top_k_precision(predictions, k=3))
print("Top 10 Precision:", top_k_precision(predictions, k=10))

Winner Precision: 0.5128205128205128
Top 3 Precision: 0.6410256410256411
Top 10 Precision: 0.7666666666666667


In [29]:
#recall@k

def top_k_recall(predictions, k):
    recall = []

    for _, group in predictions.groupby('session_id'):
        actual_top_k = set(group.nsmallest(k, 'Final_Race_Position')['Driver']) #actual ranks
        predicted_top_k = set(group.nsmallest(k, 'predicted_position')['Driver']) #predicted ranks

        correct_predictions = len(actual_top_k & predicted_top_k)
        recall.append(correct_predictions / len(actual_top_k)) 

    return np.mean(recall) #average recall across races

print("Winner Recall:", top_k_recall(predictions, k=1))
print("Top 3 Recall:", top_k_recall(predictions, k=3))
print("Top 10 Recall:", top_k_recall(predictions, k=10))

Winner Recall: 0.5128205128205128
Top 3 Recall: 0.6410256410256411
Top 10 Recall: 0.7666666666666667


In [30]:
#F1@k

def top_k_f1(predictions, k):
    f1_scores = []

    for _, group in predictions.groupby('session_id'):
        actual_top_k = set(group.nsmallest(k, 'Final_Race_Position')['Driver']) #binary top k
        y_true = group['Driver'].apply(lambda x: 1 if x in actual_top_k else 0)

        predicted_top_k = set(group.nsmallest(k, 'predicted_position')['Driver']) #predicted binary top k
        y_pred = group['Driver'].apply(lambda x: 1 if x in predicted_top_k else 0)

        score = f1_score(y_true, y_pred)
        f1_scores.append(score)

    return np.mean(f1_scores)

print("Winner F1 Score:", top_k_f1(predictions, 1))
print("Top 3 F1 Score:", top_k_f1(predictions, 3))
print("Top 10 F1 Score:", top_k_f1(predictions, 10))

Winner F1 Score: 0.5128205128205128
Top 3 F1 Score: 0.6410256410256411
Top 10 F1 Score: 0.7666666666666667


In [31]:
#Spearman's Rank Correlation
#how well the predicted ranking order matches the true ranking order

def mean_spearman(df):
    scores = []
    for _, group in predictions.groupby('session_id'):
        true_ranks = group['Final_Race_Position']
        pred_ranks = group['predicted_position']

        corr, _ = spearmanr(true_ranks, pred_ranks)
        scores.append(corr)

    return np.mean(scores) #average across races

print("Mean Spearman's Rank Correlation:", mean_spearman(predictions))

Mean Spearman's Rank Correlation: 0.6131096973202237


In [32]:
#Normalized Discounted Cumulative Gain (NDCG)
#computes how well-ordered the rankings are

def compute_ndcg(group, k=10):
    group = group.copy()
    group['relevance'] = 1 / (group['Final_Race_Position'] + 1)

    y_true = group.sort_values('Final_Race_Position').head(k)['relevance'].values.reshape(1, -1)
    y_pred = group.sort_values('predicted_position').head(k)['relevance'].values.reshape(1, -1)

    return ndcg_score(y_true, y_pred)

ndcg_scores = predictions.groupby('session_id').apply(compute_ndcg, k=10)
print("Average NDCG for all Races:", ndcg_scores.mean())

Average NDCG for all Races: 0.9098258948150707


In [33]:
#Mean Reciprocal Rank (MRR)
#computes how early the model predicts the top rank - race winner

def compute_mrr(df):
    mrr_scores = []

    for session_id, group in df.groupby('session_id'):
        group_sorted = group.sort_values('predicted_position').reset_index(drop=True)

        #index of race winner
        winner_index = group_sorted[group_sorted['Final_Race_Position']==1].index

        if len(winner_index) > 0:
            #calculating reciprocal rank
            rr = 1.0 / (winner_index[0]+1) #adding 1 because it is a 0-based index
            mrr_scores.append(rr)

    return sum(mrr_scores)/len(mrr_scores) #average of reciprocal rank scores

mrr = compute_mrr(predictions)
print("Mean Reciprocal Rank:", mrr)

Mean Reciprocal Rank: 0.6727207977207977


In [ ]:
#hyperparameter tuning using Optuna

def objective(trial):
    param = {
        'tree_method': 'hist',
        'booster': 'gbtree',
        'objective': 'rank:pairwise',
        'eval_metric': 'ndcg',
        'random_state': 42,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0)
    }
    
    model = xgb.XGBRanker(**param)

    #cross-validation
    gkf = GroupKFold(n_splits=3) 
    scores = []
    
    session_ids = train_data['session_id'].values
    for train_idx, valid_idx in gkf.split(X_train, y_train, groups=session_ids):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]
        session_tr = session_ids[train_idx]
        session_val = session_ids[valid_idx]

        group_tr = np.unique(session_tr, return_counts=True)[1]
        group_val = np.unique(session_val, return_counts=True)[1]

        model.fit(X_tr, y_tr, group=group_tr, verbose=False)

        preds = model.predict(X_val)
        
        ndcg = ndcg_score(y_val.values.reshape(1, -1), preds.reshape(1, -1))
        scores.append(ndcg)

    return np.mean(scores)

In [35]:
study = optuna.create_study(direction='maximize') #maximizing NDCG
study.optimize(objective, n_trials = 100)

[I 2025-04-29 14:18:21,693] A new study created in memory with name: no-name-8067a9c4-b766-4a00-b483-c728eada8eed
[I 2025-04-29 14:18:22,080] Trial 0 finished with value: 0.9505568996373971 and parameters: {'learning_rate': 0.03041872291022377, 'colsample_bytree': 0.7873526579170214, 'max_depth': 5, 'n_estimators': 467, 'subsample': 0.788188108556221}. Best is trial 0 with value: 0.9505568996373971.
[I 2025-04-29 14:18:22,279] Trial 1 finished with value: 0.9529853341788584 and parameters: {'learning_rate': 0.04029832156424745, 'colsample_bytree': 0.7688980348138582, 'max_depth': 8, 'n_estimators': 133, 'subsample': 0.8158421175757722}. Best is trial 1 with value: 0.9529853341788584.
[I 2025-04-29 14:18:22,427] Trial 2 finished with value: 0.9581362907658911 and parameters: {'learning_rate': 0.012283297897914158, 'colsample_bytree': 0.6579433169776243, 'max_depth': 7, 'n_estimators': 137, 'subsample': 0.7402857115037647}. Best is trial 2 with value: 0.9581362907658911.
[I 2025-04-29 14

In [36]:
trial = study.best_trial
print("Best Score:", trial.value) #best NDCG score
print("Best Params:")
for key, value in trial.params.items():
    print(key, ": ", value)

Best Score: 0.9604505887101453
Best Params:
learning_rate :  0.010583375896735903
colsample_bytree :  0.7099012039474639
max_depth :  3
n_estimators :  155
subsample :  0.9622757804952629


In [ ]:
#retrain model with optimal parameters

model = xgb.XGBRanker(
    tree_method = 'hist', #fastest algorithm
    booster = 'gbtree', #gradient boosting
    objective = 'rank:pairwise', #comparing pairs of drivers
    random_state = 42,
    learning_rate = 0.01,
    colsample_bytree = 0.71, #percentage of features used
    max_depth = 3, #depth of trees
    n_estimators = 155, 
    subsample = 0.962,
    eval_metric = 'ndcg'
)

model.fit(X_train, y_train, group=groups, verbose=True)

XGBRanker(base_score=None, booster='gbtree', callbacks=None,
          colsample_bylevel=None, colsample_bynode=None, colsample_bytree=0.71,
          device=None, early_stopping_rounds=None, enable_categorical=False,
          eval_metric='ndcg', feature_types=None, feature_weights=None,
          gamma=None, grow_policy=None, importance_type=None,
          interaction_constraints=None, learning_rate=0.01, max_bin=None,
          max_cat_threshold=None, max_cat_to_onehot=None, max_delta_step=None,
          max_depth=3, max_leaves=None, min_child_weight=None, missing=nan,
          monotone_constraints=None, multi_strategy=None, n_estimators=155,
          n_jobs=None, num_parallel_tree=None, ...)

In [38]:
def predict(model, df):
    scores = model.predict(df.drop(columns=['session_id', 'Final_Race_Position']))
    predictions_df = df.copy()
    predictions_df['predicted_score'] = scores
    predictions_df['predicted_position'] = predictions_df['predicted_score'].rank(ascending=True, method='first')
    return predictions_df

predictions2 = (
    test_data.groupby('session_id')
    .apply(lambda x: predict(model, x))
    .reset_index(drop=True)
)

In [39]:
print("-- Metrics with Optimal Parameters --")

print("Winner Precision:", top_k_precision(predictions2, k=1))
print("Top 3 Precision:", top_k_precision(predictions2, k=3))
print("Top 10 Precision:", top_k_precision(predictions2, k=10), "\n")

print("Winner Recall:", top_k_recall(predictions2, k=1))
print("Top 3 Recall:", top_k_recall(predictions2, k=3))
print("Top 10 Recall:", top_k_recall(predictions2, k=10), "\n")

print("Winner F1 Score:", top_k_f1(predictions2, 1))
print("Top 3 F1 Score:", top_k_f1(predictions2, 3))
print("Top 10 F1 Score:", top_k_f1(predictions2, 10), "\n")

print("Mean Spearman's Rank Correlation:", mean_spearman(predictions2), "\n")

ndcg_scores = predictions2.groupby('session_id').apply(compute_ndcg, k=10)
print("Average NDCG for all Races:", ndcg_scores.mean(), "\n")

print("Mean Reciprocal Rank:", compute_mrr(predictions2))

-- Metrics with Optimal Parameters --
Winner Precision: 0.5384615384615384
Top 3 Precision: 0.6752136752136753
Top 10 Precision: 0.7743589743589743 

Winner Recall: 0.5384615384615384
Top 3 Recall: 0.6752136752136753
Top 10 Recall: 0.7743589743589743 

Winner F1 Score: 0.5384615384615384
Top 3 F1 Score: 0.6752136752136753
Top 10 F1 Score: 0.7743589743589744 

Mean Spearman's Rank Correlation: 0.6131096973202237 

Average NDCG for all Races: 0.9217651743220173 

Mean Reciprocal Rank: 0.7001942501942501


In [ ]:
#saving model
joblib.dump(model, 'f1_xgbranker.pkl')

['f1_xgbranker.pkl']

In [ ]:
#saving feature order
joblib.dump(X_train.columns.tolist(), 'train_columns.joblib')

['train_columns.joblib']